# Milestone 2: Visual Analysis

Group Fusion (tfs523, jeb6694). This notebook answers the seven exploration questions from Milestone 0 with Altair charts. For each question we keep the exploratory charts, then show one refined chart and the insight we gained.

Data: `fused.parquet` (one row per airport-day, 9,416 rows at 320 contiguous-US airports) and `flights_clean.parquet` (one row per flight), both saved in Milestone 1.

## Setup

In [ ]:
!pip install -q vegafusion vl-convert-python
import pandas as pd
import numpy as np
import altair as alt
alt.data_transformers.enable('vegafusion')   # Altair refuses DataFrames over 5,000 rows without it

PROJECT_ID = 'global-pagoda-507220-k4'
BUCKET = 'gs://cs329e-fusion-datasets'

In [ ]:
from google.colab import auth
auth.authenticate_user()
!gcloud config set project {PROJECT_ID}

## Load the Milestone 1 outputs

We label each airport-day by the weather in the airport's 0.1° tile: **clear** (no lightning, no hail), **lightning only**, **hail only**, or **both**. Each flight gets the label of its origin airport on that day, through a merge on date and airport. The merge is an inner merge, so it keeps only the contiguous-US airports that `fused` covers.

In [3]:
fused = pd.read_parquet(f'{BUCKET}/processed/fused.parquet').reset_index()
flights = pd.read_parquet(f'{BUCKET}/processed/flights_clean.parquet').reset_index()

def weather_type(row):
    if row['lightning_count'] > 0 and row['hail_count'] > 0:
        return 'both'
    if row['lightning_count'] > 0:
        return 'lightning only'
    if row['hail_count'] > 0:
        return 'hail only'
    return 'clear'

fused['weather'] = fused.apply(weather_type, axis=1)
fused['storm'] = fused['weather'] != 'clear'
weather_order = ['clear', 'lightning only', 'hail only', 'both']

fused['origin'] = fused['origin'].astype(str)
flights['origin'] = flights['origin'].astype(str)
flights = flights.merge(fused[['flight_date', 'origin', 'weather']], on=['flight_date', 'origin'], how='inner')
flights['storm'] = flights['weather'] != 'clear'

causes = ['carrier_delay', 'weather_delay', 'nas_delay', 'security_delay', 'late_aircraft_delay']
print(len(fused), 'airport-days,', len(flights), 'flights')
fused['weather'].value_counts()

9416 airport-days, 588955 flights


weather
clear             7980
both               816
lightning only     317
hail only          303
Name: count, dtype: int64

## Question 1

**How does the daily volume of lightning near an airport relate to that airport's mean departure delay?**

### Exploratory charts

In [4]:
# Every airport-day. Use the mouse wheel to zoom into the crowded area near 0 strikes.
alt.Chart(fused).mark_circle(size=15, opacity=0.4).encode(
    alt.X('lightning_count:Q', title='Lightning strikes in airport tile'),
    alt.Y('mean_dep_delay:Q', title='Mean departure delay (min)'),
    tooltip=['origin', 'flight_date', 'lightning_count', 'mean_dep_delay']
).properties(
    title='Lightning Strikes vs Mean Departure Delay, Every Airport-Day',
    width=500
).interactive()

alt.Chart(...)

In [5]:
# The scatter is dominated by the 8,283 days with 0 strikes, so group the strike counts into levels
def lightning_level(n):
    if n == 0:
        return '0'
    if n <= 10:
        return '1-10'
    if n <= 50:
        return '11-50'
    if n <= 200:
        return '51-200'
    return '200+'

level_order = ['0', '1-10', '11-50', '51-200', '200+']
fused['lightning_level'] = fused['lightning_count'].apply(lightning_level)

alt.Chart(fused).mark_line(point=True).encode(
    alt.X('lightning_level:O', title='Lightning strikes in airport tile', sort=level_order),
    alt.Y('mean(mean_dep_delay):Q', title='Mean departure delay (min)')
).properties(
    title='Mean Departure Delay by Lightning Level',
    width=400
)

alt.Chart(...)

In [6]:
q1 = fused.groupby('lightning_level').agg(
    airport_days=('mean_dep_delay', 'size'),
    mean_dep_delay=('mean_dep_delay', 'mean'),
    cancel_rate=('cancel_rate', 'mean')
).reset_index()
q1['delay_label'] = q1['mean_dep_delay'].round(1).astype(str)
q1

,lightning_level,airport_days,mean_dep_delay,cancel_rate,delay_label
0,0,8283,16.610366,0.017062,16.6
1,1-10,836,24.334525,0.038343,24.3
2,11-50,235,27.188750,0.047304,27.2
3,200+,6,66.447706,0.180100,66.4
4,51-200,56,57.015154,0.041400,57.0


### Refined chart

Design changes: an active title that states the finding, bars in strike order, a sequential color that gets darker as lightning increases, a value label on each bar, and a tooltip with the number of airport-days behind each bar. The 200+ bar rests on only 6 airport-days, which the tooltip shows.

In [7]:
bars = alt.Chart(q1).mark_bar().encode(
    alt.X('lightning_level:O', title='Lightning strikes in the airport tile that day', sort=level_order, axis=alt.Axis(labelAngle=0)),
    alt.Y('mean_dep_delay:Q', title='Mean departure delay (min)'),
    alt.Color('lightning_level:O', sort=level_order, scale=alt.Scale(scheme='orangered'), legend=None),
    tooltip=['lightning_level', 'airport_days',
             alt.Tooltip('mean_dep_delay:Q', format='.1f'), alt.Tooltip('cancel_rate:Q', format='.1%')]
)
labels = alt.Chart(q1).mark_text(dy=-8).encode(
    alt.X('lightning_level:O', sort=level_order),
    alt.Y('mean_dep_delay:Q'),
    text='delay_label'
)
(bars + labels).properties(
    title='Departure Delays Quadruple as Lightning at the Airport Increases',
    width=450,
    height=300
)

alt.LayerChart(...)

### Insight

Departure delay rises steadily with lightning at the airport: from 16.6 minutes on days with no strikes to 24.3 minutes with 1-10 strikes and 57.0 minutes above 50 strikes. The rise is not a straight line, because a little lightning adds a few minutes but heavy lightning adds tens of minutes, probably because ramp crews must stop work. The refined bar chart is the most effective version, because the scatter is hidden under 8,283 days with no lightning and the line chart does not show how few days are in the top levels.

## Question 2

**Which of the 351 airports show the strongest weather-delay sensitivity, and does that track their geography?**

We define an airport's *storm sensitivity* as its mean departure delay on days with lightning in its tile minus its mean delay on days without. To keep the numbers stable, we keep airports with at least 4 lightning days and at least 30 departures on an average day without lightning. This leaves 50 airports.

In [8]:
lightning_days = fused[fused['lightning_count'] > 0].groupby('origin').agg(
    storm_delay=('mean_dep_delay', 'mean'),
    lightning_days=('mean_dep_delay', 'size'))
other_days = fused[fused['lightning_count'] == 0].groupby('origin').agg(
    calm_delay=('mean_dep_delay', 'mean'),
    flights_per_day=('n_flights', 'mean'))
location = fused.groupby('origin').agg(state=('origin_state', 'first'), lat=('lat', 'first'), lon=('lon', 'first'))

q2 = lightning_days.join(other_days).join(location).reset_index()
q2 = q2[(q2['lightning_days'] >= 4) & (q2['flights_per_day'] >= 30)]
q2['sensitivity'] = q2['storm_delay'] - q2['calm_delay']
q2.sort_values('sensitivity', ascending=False).head(10)

,origin,storm_delay,lightning_days,calm_delay,flights_per_day,state,lat,lon,sensitivity
71,DFW,74.554499,4,25.111204,928.192308,TX,32.9,-97.0,49.443296
211,ROC,44.384303,4,12.827735,30.730769,NY,43.1,-77.7,31.556569
65,DAL,55.076544,4,25.594933,200.692308,TX,32.8,-96.8,29.481611
125,IAH,45.105921,7,15.859079,331.521739,TX,30.0,-95.3,29.246842
221,SFB,46.664628,12,18.419477,32.833333,FL,28.8,-81.2,28.245151
217,SAV,39.068210,6,11.263943,54.750000,GA,32.1,-81.2,27.804267
148,LIT,39.336826,5,13.496200,42.040000,AR,34.7,-92.2,25.840625
179,MSY,38.613850,7,17.058135,125.000000,LA,30.0,-90.3,21.555716
161,MDW,40.406662,5,19.380780,229.120000,IL,41.8,-87.8,21.025881
176,MSN,32.762634,5,11.995672,43.520000,WI,43.1,-89.3,20.766962


### Exploratory charts

In [9]:
alt.Chart(q2).mark_bar().encode(
    alt.X('origin:N', title='Airport', sort='-y'),
    alt.Y('sensitivity:Q', title='Extra delay on lightning days (min)'),
    tooltip=['origin', 'state', 'lightning_days']
).properties(
    title='Storm Sensitivity by Airport',
    width=700
)

alt.Chart(...)

In [10]:
# Median sensitivity by state: is there a regional pattern?
alt.Chart(q2).mark_bar().encode(
    alt.X('state:N', title='State', sort='-y'),
    alt.Y('median(sensitivity):Q', title='Median extra delay on lightning days (min)'),
    tooltip=['state', 'count()']
).properties(
    title='Storm Sensitivity by State',
    width=600
)

alt.Chart(...)

### Refined chart

Design changes: a bubble map with longitude on x and latitude on y, as in the flu map from class, so the airports form the shape of the US with no base map. Circle size encodes airport traffic and a sequential color encodes the extra delay. The five most sensitive airports are labeled (DAL is labeled below its circle, because it sits next to DFW). Interaction: click an airport to highlight every airport in the same state, and the title states the finding.

In [11]:
state_select = alt.selection_point(fields=['state'])

points = alt.Chart(q2).mark_circle(stroke='gray').encode(
    alt.X('lon:Q', title='Longitude', scale=alt.Scale(domain=[-125, -66])),
    alt.Y('lat:Q', title='Latitude', scale=alt.Scale(domain=[24, 50])),
    alt.Size('flights_per_day:Q', title='Departures per day'),
    alt.Color('sensitivity:Q', title='Extra delay (min)', scale=alt.Scale(scheme='orangered')),
    opacity=alt.condition(state_select, alt.value(0.9), alt.value(0.15)),
    tooltip=['origin', 'state', alt.Tooltip('sensitivity:Q', format='.1f'), 'lightning_days', 'flights_per_day']
).add_params(state_select)

top5 = q2.sort_values('sensitivity', ascending=False).head(5)
label_above = alt.Chart(top5[top5['origin'] != 'DAL']).mark_text(dy=-14, fontWeight='bold').encode(
    alt.X('lon:Q'), alt.Y('lat:Q'), text='origin')
label_below = alt.Chart(top5[top5['origin'] == 'DAL']).mark_text(dy=16, fontWeight='bold').encode(
    alt.X('lon:Q'), alt.Y('lat:Q'), text='origin')

(points + label_above + label_below).properties(
    title='Texas Hubs Lose the Most Time to Lightning; Florida, with the Most Storm Days, Loses the Least',
    width=700,
    height=400
)

alt.LayerChart(...)

### Insight

Storm sensitivity is highest at the big Texas hubs: DFW loses 49 extra minutes on lightning days, and DAL and IAH about 29 each, although DFW's number rests on only 4 lightning days. Florida has the most lightning days of any state (MCO 12, ECP 14), but its 12 airports in this set have a median increase of only 5.5 minutes, against 13.0 elsewhere (SFB, at +28, is the exception). So geography matters, but in an unexpected direction: where storms are routine, airlines seem to plan around them, while rarer storms at a busy hub cause a chain of delays. The bubble map is the most effective version, because the bar charts hide location and the state chart averages over very different airports.

## Question 3

**Does BTS's own WeatherDelay attribution agree with independently observed lightning and hail activity on the same date and location?**

We compare two yes/no flags per airport-day: did BTS record any weather-delay minutes, and did the airport's tile have any lightning or hail.

In [12]:
fused['bts'] = np.where(fused['weather_delay'] > 0, 'BTS weather delay', 'No BTS weather delay')
fused['observed'] = np.where(fused['storm'], 'Lightning or hail in tile', 'No lightning or hail')

q3 = fused.groupby(['bts', 'observed']).agg(
    airport_days=('weather_delay', 'size'),
    minutes=('weather_delay', 'sum')
).reset_index()
row_totals = q3.groupby('bts').agg(row_total=('airport_days', 'sum')).reset_index()
q3 = q3.merge(row_totals, on='bts')
q3['row_share'] = q3['airport_days'] / q3['row_total']
q3

,bts,observed,airport_days,minutes,row_total,row_share
0,BTS weather delay,Lightning or hail in tile,651,391937.0,2395,0.271816
1,BTS weather delay,No lightning or hail,1744,373375.0,2395,0.728184
2,No BTS weather delay,Lightning or hail in tile,785,0.0,7021,0.111807
3,No BTS weather delay,No lightning or hail,6236,0.0,7021,0.888193


### Exploratory charts

In [13]:
alt.Chart(q3).mark_bar().encode(
    alt.X('observed:N', title=None),
    alt.Y('airport_days:Q', title='Airport-days'),
    alt.Color('observed:N', title='Observed weather'),
    column='bts:N'
).properties(
    title='Airport-Days by BTS Attribution and Observed Weather',
    width=200
)

alt.Chart(...)

In [14]:
alt.Chart(q3[q3['minutes'] > 0]).mark_bar().encode(
    alt.X('observed:N', title=None),
    alt.Y('minutes:Q', title='BTS weather-delay minutes'),
    alt.Color('observed:N', legend=None)
).properties(
    title='BTS Weather-Delay Minutes by Observed Weather',
    width=300
)

alt.Chart(...)

### Refined chart

Design changes: a 2 x 2 agreement matrix (a heat map), so agreement sits on one diagonal and disagreement on the other. Each cell shows its count and its share of the row, the color encodes the row share, the text color switches to white on dark cells so it stays readable, and the title states the finding.

In [15]:
q3['label'] = q3.apply(lambda row: f"{row['airport_days']:,} ({row['row_share']:.0%})", axis=1)

heat = alt.Chart(q3).mark_rect().encode(
    alt.X('observed:N', title='Observed in the airport tile', axis=alt.Axis(labelAngle=0)),
    alt.Y('bts:N', title='BTS attribution'),
    alt.Color('row_share:Q', title='Share of row', scale=alt.Scale(scheme='lighttealblue', domain=[0, 1])),
    tooltip=['bts', 'observed', 'airport_days', 'minutes']
)
text = alt.Chart(q3).mark_text(fontSize=14).encode(
    alt.X('observed:N'),
    alt.Y('bts:N'),
    text='label',
    color=alt.condition(alt.datum.row_share > 0.5, alt.value('white'), alt.value('black'))
)
(heat + text).properties(
    title='73% of Airport-Days with a BTS Weather Delay Had No Storm in the Airport Tile',
    width=420,
    height=220
)

alt.LayerChart(...)

### Insight

The two sources agree only in part. Of the 2,395 airport-days with BTS weather-delay minutes, 1,744 (73%) had no lightning or hail in the airport's tile, and those days carry 49% of all weather-delay minutes. This is expected: BTS records weather delay for storms en route or at the destination, while our tile is only about 11 x 9 km around the origin airport. The agreement matrix is the most effective version, because it shows agreement and disagreement side by side, while the bar charts hide the row shares.

## Question 4

**How do the five delay causes shift in composition on days with severe weather versus clear days?**

We sum each cause's minutes by weather type and divide by the number of scheduled flights. This makes the weather types comparable even though they have very different numbers of days.

In [16]:
q4_wide = fused.groupby('weather').agg(
    flights=('n_flights', 'sum'),
    carrier_delay=('carrier_delay', 'sum'),
    weather_delay=('weather_delay', 'sum'),
    nas_delay=('nas_delay', 'sum'),
    security_delay=('security_delay', 'sum'),
    late_aircraft_delay=('late_aircraft_delay', 'sum')
).reset_index()

# Altair needs one row per (weather, cause): make one small data frame per cause and stack them vertically
parts = []
for cause in causes:
    parts.append(pd.DataFrame({'weather': q4_wide['weather'],
                               'cause': cause.replace('_delay', '').replace('_', ' '),
                               'minutes_per_flight': q4_wide[cause] / q4_wide['flights']}))
q4 = pd.concat(parts)
q4_totals = q4.groupby('weather').agg(total=('minutes_per_flight', 'sum')).reset_index()
q4_totals['total_label'] = q4_totals['total'].round(1).astype(str)
q4_totals

,weather,total,total_label
0,both,35.003855,35.0
1,clear,17.030707,17.0
2,hail only,24.813941,24.8
3,lightning only,24.590761,24.6


### Exploratory charts

In [17]:
alt.Chart(q4).mark_bar().encode(
    alt.X('weather:N', title='Weather at the airport', sort=weather_order),
    alt.Y('minutes_per_flight:Q', title='Delay minutes per flight'),
    alt.Color('cause:N', title='Delay cause')
).properties(
    title='Delay Minutes per Flight by Cause and Weather',
    width=400
)

alt.Chart(...)

In [18]:
alt.Chart(q4).mark_bar().encode(
    alt.X('weather:N', title=None, sort=weather_order),
    alt.Y('minutes_per_flight:Q', title='Delay minutes per flight'),
    alt.Color('weather:N', sort=weather_order, legend=None),
    column='cause:N'
).properties(
    title='Each Delay Cause by Weather',
    width=120
)

alt.Chart(...)

### Refined chart

Design changes: horizontal stacked bars in weather order, with purposeful color: weather delay in red and late-aircraft delay in dark blue stand out, and the other causes are muted grays. Each bar is labeled with its total, and the title states the finding.

In [19]:
cause_colors = alt.Scale(domain=['weather', 'late aircraft', 'nas', 'carrier', 'security'],
                         range=['#c03d3e', '#1f4e79', '#9db4c8', '#c9c9c9', '#e8e8e8'])
bars = alt.Chart(q4).mark_bar().encode(
    alt.Y('weather:N', title=None, sort=weather_order),
    alt.X('minutes_per_flight:Q', title='Delay minutes per scheduled flight'),
    alt.Color('cause:N', title='Cause', scale=cause_colors),
    tooltip=['weather', 'cause', alt.Tooltip('minutes_per_flight:Q', format='.2f')]
)
totals = alt.Chart(q4_totals).mark_text(dx=16).encode(
    alt.Y('weather:N', sort=weather_order),
    alt.X('total:Q'),
    text='total_label'
)
(bars + totals).properties(
    title='On Lightning-and-Hail Days, Late Aircraft Add More Delay than the Storm Itself',
    width=500,
    height=200
)

alt.LayerChart(...)

### Insight

Severe weather raises every cause, not only weather delay. On days with both lightning and hail, total delay per flight is about twice the clear-day level (35.0 vs 17.0 minutes). Weather delay grows almost ninefold (0.7 to 6.5 minutes), but late-aircraft delay grows more in minutes (7.1 to 15.0), and NAS delay also nearly doubles. The refined stacked bar is the most effective version, because it shows both the total and the mix in one bar, while the faceted chart makes the reader add up five panels.

## Question 5

**Do hail days and lightning days produce different operational responses (cancellations versus delays)?**

Here we work at the flight level: each flight carries the weather label of its origin airport on that day.

In [20]:
flights['weather_cancel'] = flights['cancellation_code'] == 'B'   # B = cancelled for weather
q5 = flights.groupby('weather').agg(
    flights=('cancelled', 'size'),
    cancel_rate=('cancelled', 'mean'),
    weather_cancel_rate=('weather_cancel', 'mean'),
    mean_dep_delay=('dep_delay', 'mean')
).reset_index()
q5

,weather,flights,cancel_rate,weather_cancel_rate,mean_dep_delay
0,both,49540,0.059830,0.046023,35.436251
1,clear,501254,0.012173,0.007649,15.991307
2,hail only,22466,0.022167,0.010727,22.647358
3,lightning only,15695,0.021153,0.014272,23.023032


### Exploratory charts

In [21]:
cancel = alt.Chart(q5).mark_bar().encode(
    alt.X('weather:N', title=None, sort=weather_order),
    alt.Y('cancel_rate:Q', title='Cancellation rate', axis=alt.Axis(format='%'))
).properties(title='Cancellation Rate by Weather', width=250)

delay = alt.Chart(q5).mark_bar().encode(
    alt.X('weather:N', title=None, sort=weather_order),
    alt.Y('mean_dep_delay:Q', title='Mean departure delay (min)')
).properties(title='Mean Departure Delay by Weather', width=250)

cancel | delay

alt.HConcatChart(...)

### Refined chart

Design changes: one scatter chart that shows both responses (delay on x, cancellation rate on y), so each weather type is one point and the mix of responses can be read at once. Direct labels next to each point instead of a legend (the "lightning only" label sits below its point, because its point overlaps "hail only"), and an active title.

In [22]:
points = alt.Chart(q5).mark_circle(size=400).encode(
    alt.X('mean_dep_delay:Q', title='Mean departure delay (min)', scale=alt.Scale(domain=[10, 40])),
    alt.Y('cancel_rate:Q', title='Share of flights cancelled', axis=alt.Axis(format='%'), scale=alt.Scale(domain=[0, 0.07])),
    alt.Color('weather:N', sort=weather_order, legend=None),
    tooltip=['weather', 'flights', alt.Tooltip('cancel_rate:Q', format='.1%'),
             alt.Tooltip('weather_cancel_rate:Q', format='.1%'), alt.Tooltip('mean_dep_delay:Q', format='.1f')]
)
names = alt.Chart(q5[q5['weather'] != 'lightning only']).mark_text(align='left', dx=14, fontSize=12).encode(
    alt.X('mean_dep_delay:Q'), alt.Y('cancel_rate:Q'), text='weather')
lightning_name = alt.Chart(q5[q5['weather'] == 'lightning only']).mark_text(align='left', dx=10, dy=18, fontSize=12).encode(
    alt.X('mean_dep_delay:Q'), alt.Y('cancel_rate:Q'), text='weather')

(points + names + lightning_name).properties(
    title='Hail Alone and Lightning Alone Look Alike; Together They Cancel 6% of Flights',
    width=450,
    height=320
)

alt.LayerChart(...)

### Insight

Hail days and lightning days produce almost the same response: about 2.2% and 2.1% of flights cancelled and 22.6 and 23.0 minutes of mean delay. The big step comes when both occur at once, with 6.0% cancelled (five times the clear-day rate of 1.2%) and 35.4 minutes of delay, which suggests that the combination marks the strongest storms. The scatter is the most effective version, because the two separate bar charts make the reader match the two responses by eye.

## Question 6

**How far does weather disruption propagate through LateAircraftDelay to airports with no local severe weather?**

For each day we measure national storm activity (the total lightning strikes across all airport tiles) and the late-aircraft delay per flight at the airports that had **no** lightning or hail that day.

In [23]:
national = fused.groupby('flight_date').agg(
    lightning_total=('lightning_count', 'sum'),
    storm_airports=('storm', 'sum')
).reset_index()
dry = fused[fused['storm'] == False].groupby('flight_date').agg(
    late_aircraft=('late_aircraft_delay', 'sum'),
    dry_flights=('n_flights', 'sum')
).reset_index()

q6 = national.merge(dry, on='flight_date')
q6['late_aircraft_per_flight'] = q6['late_aircraft'] / q6['dry_flights']
q6.describe()

,flight_date,lightning_total,storm_airports,late_aircraft,dry_flights,late_aircraft_per_flight
count,30,30.000000,30.000000,30.000000,30.000000,30.000000
mean,2026-06-15 12:00:00,480.066667,47.866667,118858.700000,16708.466667,7.184087
min,2026-06-01 00:00:00,61.000000,21.000000,46596.000000,13648.000000,2.965631
25%,2026-06-08 06:00:00,300.000000,37.750000,89848.000000,15469.500000,4.934071
50%,2026-06-15 12:00:00,437.000000,47.000000,103486.500000,16966.500000,6.950104
75%,2026-06-22 18:00:00,557.750000,56.750000,149004.250000,17634.750000,8.524894
max,2026-06-30 00:00:00,1559.000000,78.000000,211468.000000,19835.000000,12.867682
std,NaN,302.936767,15.736479,43194.862628,1596.201390,2.772771


### Exploratory charts

In [24]:
storms = alt.Chart(q6).mark_line(color='firebrick').encode(
    alt.X('flight_date:T', title='Date'),
    alt.Y('storm_airports:Q', title='Airports with lightning or hail')
).properties(width=600, height=150)

late = alt.Chart(q6).mark_line().encode(
    alt.X('flight_date:T', title='Date'),
    alt.Y('late_aircraft_per_flight:Q', title='Late-aircraft min/flight, dry airports')
).properties(width=600, height=150)

(storms & late).properties(title='Storm Airports and Late-Aircraft Delay at Dry Airports, by Day')

alt.VConcatChart(...)

In [25]:
alt.Chart(q6).mark_circle(size=60).encode(
    alt.X('lightning_total:Q', title='Lightning strikes across all airport tiles that day'),
    alt.Y('late_aircraft_per_flight:Q', title='Late-aircraft min/flight, dry airports'),
    tooltip=['flight_date', 'lightning_total', alt.Tooltip('late_aircraft_per_flight:Q', format='.1f')]
).properties(
    title='National Lightning vs Late-Aircraft Delay at Dry Airports',
    width=450
)

alt.Chart(...)

### Refined chart

The scatter shows an upward trend but is noisy, so we group the 30 days into three levels of national lightning and compare the means. Design changes: bars in level order, a sequential color for the lightning level, value labels, a tooltip with the number of days in each level, and an active title.

In [26]:
def national_level(total):
    if total < 300:
        return 'Low (under 300 strikes)'
    if total < 600:
        return 'Medium (300-600)'
    return 'High (over 600)'

national_order = ['Low (under 300 strikes)', 'Medium (300-600)', 'High (over 600)']
q6['national_level'] = q6['lightning_total'].apply(national_level)
q6_levels = q6.groupby('national_level').agg(
    days=('flight_date', 'size'),
    late_aircraft_per_flight=('late_aircraft_per_flight', 'mean')
).reset_index()
q6_levels['label'] = q6_levels['late_aircraft_per_flight'].round(1).astype(str)

bars = alt.Chart(q6_levels).mark_bar().encode(
    alt.X('national_level:O', title='Lightning across all airport tiles that day', sort=national_order, axis=alt.Axis(labelAngle=0)),
    alt.Y('late_aircraft_per_flight:Q', title='Late-aircraft delay per flight (min)'),
    alt.Color('national_level:O', sort=national_order, scale=alt.Scale(scheme='blues'), legend=None),
    tooltip=['national_level', 'days', alt.Tooltip('late_aircraft_per_flight:Q', format='.2f')]
)
labels = alt.Chart(q6_levels).mark_text(dy=-8).encode(
    alt.X('national_level:O', sort=national_order),
    alt.Y('late_aircraft_per_flight:Q'),
    text='label'
)
(bars + labels).properties(
    title='Storms Elsewhere Delay Airports That Stay Dry',
    width=450,
    height=300
)

alt.LayerChart(...)

### Insight

Disruption does travel. At airports with no local lightning or hail, late-aircraft delay averages 5.9 minutes per flight on low-lightning days, 6.6 on medium days and 9.8 on high days (8, 15 and 7 days). So late inbound aircraft carry storm delay to dry airports, and a local weather flag alone undercounts the cost of storms. The grouped bar chart is the most effective version, because the scatter and the two time series show the same pattern but make the reader find it in noisy day-to-day points.

## Question 7

**Which carriers absorb weather disruption best, controlling for their route geography?**

Carriers fly to different airports, so a raw comparison mixes the airline with its network. To control for geography, we compare each carrier with itself at the same airport. For every carrier-airport pair with at least 20 departures on storm days and 20 on clear days, we take the storm-day mean delay minus the clear-day mean delay. Then we average those differences for each carrier, weighted by the number of storm-day departures.

In [27]:
names = {'AA': 'American', 'AS': 'Alaska', 'B6': 'JetBlue', 'DL': 'Delta', 'F9': 'Frontier', 'G4': 'Allegiant',
         'MQ': 'Envoy', 'OH': 'PSA', 'OO': 'SkyWest', 'UA': 'United', 'WN': 'Southwest', 'YX': 'Republic'}
flights['airline'] = flights['airline'].astype(str)
flights['carrier'] = flights['airline'].map(names) + ' (' + flights['airline'] + ')'

storm_pairs = flights[flights['storm']].groupby(['carrier', 'origin']).agg(
    storm_delay=('dep_delay', 'mean'), storm_flights=('dep_delay', 'size')).reset_index()
clear_pairs = flights[flights['storm'] == False].groupby(['carrier', 'origin']).agg(
    clear_delay=('dep_delay', 'mean'), clear_flights=('dep_delay', 'size')).reset_index()

pairs = storm_pairs.merge(clear_pairs, on=['carrier', 'origin'])
pairs = pairs[(pairs['storm_flights'] >= 20) & (pairs['clear_flights'] >= 20)]
pairs['weighted_diff'] = (pairs['storm_delay'] - pairs['clear_delay']) * pairs['storm_flights']

q7 = pairs.groupby('carrier').agg(
    weighted_diff=('weighted_diff', 'sum'),
    storm_flights=('storm_flights', 'sum'),
    airports=('origin', 'size')
).reset_index()
q7['within_airport'] = q7['weighted_diff'] / q7['storm_flights']

raw = flights.groupby(['carrier', 'storm']).agg(delay=('dep_delay', 'mean')).reset_index()
raw['day'] = np.where(raw['storm'], 'storm days', 'clear days')
q7.sort_values('within_airport')

,carrier,weighted_diff,storm_flights,airports,within_airport
9,SkyWest (OO),61590.733302,7053,68,8.732558
6,JetBlue (B6),30919.372170,3178,26,9.729192
1,Allegiant (G4),21813.998015,2054,30,10.620252
10,Southwest (WN),198239.737651,17532,70,11.307309
4,Envoy (MQ),62177.744562,4759,55,13.065296
3,Delta (DL),177359.656509,12445,77,14.251479
0,Alaska (AS),10244.071343,686,16,14.933049
11,United (UA),166251.112891,10687,47,15.556387
8,Republic (YX),84927.446383,5230,40,16.238517
5,Frontier (F9),47885.492044,2758,25,17.362397


### Exploratory charts

In [28]:
# Raw mean delay on clear and storm days for each carrier
alt.Chart(raw).mark_point(size=80, filled=True).encode(
    alt.X('delay:Q', title='Mean departure delay (min)'),
    alt.Y('carrier:N', title=None),
    alt.Color('day:N', title=None),
    tooltip=['carrier', 'day', alt.Tooltip('delay:Q', format='.1f')]
).properties(
    title='Mean Departure Delay on Clear vs Storm Days, by Carrier (Raw)',
    width=450
)

alt.Chart(...)

In [29]:
raw_increase = raw[raw['storm']].merge(raw[raw['storm'] == False], on='carrier', suffixes=('_storm', '_clear'))
raw_increase['increase'] = raw_increase['delay_storm'] - raw_increase['delay_clear']

alt.Chart(raw_increase).mark_bar().encode(
    alt.X('carrier:N', title=None, sort='-y'),
    alt.Y('increase:Q', title='Extra delay on storm days (min)')
).properties(
    title='Raw Storm-Day Delay Increase by Carrier',
    width=500
)

alt.Chart(...)

### Refined chart

Design changes: the geography-controlled measure instead of the raw one, horizontal bars sorted from best to worst, and purposeful color: carriers above the all-carrier average in red and below it in blue. The average is written in the title, and the tooltip shows how many airports each estimate uses.

In [30]:
average = float(q7['weighted_diff'].sum() / q7['storm_flights'].sum())   # plain float, so Altair can write it into the chart

alt.Chart(q7).mark_bar().encode(
    alt.Y('carrier:N', title=None, sort='x'),
    alt.X('within_airport:Q', title='Extra departure delay on storm days at the same airports (min)'),
    color=alt.condition(alt.datum.within_airport > average, alt.value('#c03d3e'), alt.value('#3b75af')),
    tooltip=['carrier', alt.Tooltip('within_airport:Q', format='.1f'), 'airports', 'storm_flights']
).properties(
    title=f'SkyWest and JetBlue Absorb Storms Best; American and PSA Worst (Average {average:.1f} min)',
    width=500,
    height=320
)

alt.Chart(...)

### Insight

When we compare each carrier with itself at the same airports, storm days add 8.7 minutes of delay for SkyWest and 9.7 for JetBlue, but 19.3 for American and 22.5 for PSA, which flies regional routes for American. So American's network absorbs storms worst, and the raw comparison understates this for American (17.2 minutes raw). The refined chart is the most effective version, because the raw charts mix each carrier's resilience with the airports where it flies.